In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout ,LayerNormalization
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from itertools import cycle
import random

In [ ]:
df1=pd.read_csv('symtoms_df.csv')
df2=pd.read_csv('Symptom-severity.csv')

In [ ]:
df1.head()

In [ ]:
df2.head()

In [ ]:
df1['Symptom_4'].fillna('',inplace=True)

In [ ]:
df1['Symptoms']=df1['Symptom_1']+','+df1['Symptom_2']+','+df1['Symptom_3']+','+df1['Symptom_4']

In [ ]:
df1=df1[['Symptoms','Disease']]

In [ ]:
df1['Symptoms']=df1['Symptoms'].str.replace('_',' ')

In [ ]:
df1.tail()

In [ ]:
df2.tail()

In [ ]:
# Convert df2 into a dictionary: symptom -> weight
severity_dict = dict(zip(df2['Symptom'], df2['weight']))

In [ ]:
# --------------------------------------
# 3. Preprocess Symptoms from df1
# --------------------------------------
def preprocess_symptoms(symptom_str):
    """
    Converts comma-separated symptoms into a list of standardized tokens.
    1) Lowercase
    2) Trim spaces
    3) Replace inner spaces with underscores
    """
    # Split on commas
    symptoms = symptom_str.lower().split(',')
    # Clean each symptom token
    symptoms = [s.strip().replace(' ', '_') for s in symptoms]
    return symptoms

In [ ]:
# Create a new column in df1 with the list of symptoms
df1['Symptom_list'] = df1['Symptoms'].apply(preprocess_symptoms)

In [ ]:
# ----------------------------------------------------
# 4. Tokenize the Symptoms (build a vocabulary)
# ----------------------------------------------------
# Flatten all symptom tokens to build a complete vocabulary
all_symptoms = [sym for row in df1['Symptom_list'] for sym in row]

tokenizer = Tokenizer(lower=True, filters='')  # no filters, since we've already cleaned
tokenizer.fit_on_texts(all_symptoms)


In [ ]:
# ----------------------------------------------------
# 5. Convert Each List of Symptoms into Integer Sequences
# ----------------------------------------------------
# For each row, convert the list of symptom tokens into their integer IDs
df1['Symptom_seq'] = df1['Symptom_list'].apply(lambda x: tokenizer.texts_to_sequences(x))
# Flatten the list-of-lists for each row
df1['Symptom_seq'] = df1['Symptom_seq'].apply(lambda seq: [item for sublist in seq for item in sublist])


In [ ]:
def shuffle_symptoms(row):
    """
    Takes a row with 'Symptoms', 'Symptom_list', and 'Symptom_seq'.
    Shuffles the symptom tokens (and corresponding token IDs) so that
    the mapping between symptom and token ID remains consistent.
    Returns new columns with the shuffled data.
    """
    # Pair each token with its corresponding token ID
    pairs = list(zip(row["Symptom_list"], row["Symptom_seq"]))

    # Shuffle the (token, token_id) pairs in place
    random.shuffle(pairs)

    # Unzip the shuffled pairs back into separate lists
    shuffled_symptoms, shuffled_seq = zip(*pairs)

    # Reconstruct the "Symptoms" string by joining shuffled tokens with commas
    shuffled_symptoms_str = ", ".join(shuffled_symptoms)

    # Return a Series containing the shuffled versions
    return pd.Series({
        "Shuffled_Symptoms": shuffled_symptoms_str,
        "Shuffled_Symptom_list": list(shuffled_symptoms),
        "Shuffled_Symptom_seq": list(shuffled_seq)
    })

# Apply the shuffle function row-by-row
df1[["Shuffled_Symptoms", "Shuffled_Symptom_list", "Shuffled_Symptom_seq"]] = df1.apply(shuffle_symptoms, axis=1)

# Display the Disease column as well as original and shuffled columns
df1[[
    "Symptoms", "Disease", "Symptom_list", "Symptom_seq",
    "Shuffled_Symptoms", "Shuffled_Symptom_list", "Shuffled_Symptom_seq"
]].head()
df1=df1.drop(columns={'Symptoms','Symptom_list','Symptom_seq'})

In [ ]:
# ----------------------------------------------------
# 6. Pad the Sequences
# ----------------------------------------------------
max_length = max(df1['Shuffled_Symptom_seq'].apply(len))  # maximum length of any symptom sequence
X = pad_sequences(df1['Shuffled_Symptom_seq'], maxlen=max_length, padding='post', dtype='int32')



In [ ]:
# ----------------------------------------------------
# 7. Encode Disease Labels
# ----------------------------------------------------
le = LabelEncoder()
y = le.fit_transform(df1['Disease'])
# Convert to one-hot vectors for a multi-class classification problem
y = tf.keras.utils.to_categorical(y)


In [ ]:
# ----------------------------------------------------
# 8. Build a Weight Vector for the Vocabulary
# ----------------------------------------------------
# We'll create a numpy array, where each index corresponds to a token ID.
# If a token is found in severity_dict, use that weight; otherwise, default to 1.
vocab_size = len(tokenizer.word_index)
embedding_scaling = np.ones(vocab_size + 1, dtype=np.float32)  # +1 because index 0 is reserved (padding)

for token, index in tokenizer.word_index.items():
    if token in severity_dict:
        embedding_scaling[index] = severity_dict[token]
    else:
        embedding_scaling[index] = 1.0  # default severity weight

# Convert it to a constant tensor so it won't be trainable
embedding_scaling = tf.constant(embedding_scaling)


In [ ]:
# ----------------------------------------------------
# 9. Create a Custom Weighted Embedding Layer
# ----------------------------------------------------
import tensorflow as tf

class WeightedEmbedding(tf.keras.layers.Layer):
    """
    Custom embedding layer that scales each token's embedding vector
    by its corresponding severity weight.
    """
    def __init__(self, input_dim, output_dim, embedding_scaling, input_length, **kwargs):
        super(WeightedEmbedding, self).__init__(**kwargs)
        self.input_dim = input_dim        # Vocabulary size (+1 for padding)
        self.output_dim = output_dim      # Embedding dimension
        self.input_length = input_length
        
        # If embedding_scaling is not already a list or numpy array,
        # try to convert it to a Python list for serialization.
        if isinstance(embedding_scaling, tf.Tensor):
            self.embedding_scaling = embedding_scaling.numpy().tolist()
        else:
            self.embedding_scaling = embedding_scaling

        # Standard Keras embedding layer
        self.embedding = tf.keras.layers.Embedding(
            input_dim=self.input_dim,
            output_dim=self.output_dim,
            input_length=self.input_length,
            mask_zero=False
        )
    
    def call(self, inputs):
        # Force the inputs to int32 (needed for tf.gather)
        inputs = tf.cast(inputs, tf.int32)
        
        # Get embeddings for the input tokens; shape: (batch_size, sequence_length, output_dim)
        embeddings = self.embedding(inputs)
        
        # Gather the severity weights for each token; shape: (batch_size, sequence_length)
        # We need to convert our stored list back to a tensor.
        scaling_tensor = tf.convert_to_tensor(self.embedding_scaling, dtype=tf.float32)
        weights = tf.gather(scaling_tensor, inputs)
        
        # Expand the last dimension so that weights shape becomes (batch_size, sequence_length, 1)
        weights = tf.expand_dims(weights, axis=-1)
        
        # Multiply each embedding vector by its severity weight
        weighted_embeddings = embeddings * weights
        return weighted_embeddings
    
    def compute_output_shape(self, input_shape):
        # The output shape is (batch_size, sequence_length, output_dim)
        return input_shape + (self.output_dim,)
    
    def get_config(self):
        # Get the base config from the parent class.
        config = super(WeightedEmbedding, self).get_config()
        # Update the config with the parameters of this layer.
        config.update({
            'input_dim': self.input_dim,
            'output_dim': self.output_dim,
            'embedding_scaling': self.embedding_scaling,
            'input_length': self.input_length,
            
        })
        return config


In [ ]:
df1.tail()

In [ ]:
df1.head()

In [ ]:
# Increase the number of LSTM units and reduce dropout rates
embedding_dim =50
model = Sequential()
model.add(WeightedEmbedding(
    input_dim=vocab_size + 1,
    output_dim=embedding_dim,
    embedding_scaling=embedding_scaling,
    input_length=max_length
))

# Increased units in first LSTM, lowered dropout slightly 

model.add(LSTM(256, return_sequences=True, dropout=0.1, recurrent_dropout=0.1))

# Add an extra LSTM layer for deeper representation

model.add(LSTM(128, return_sequences=True, dropout=0.1, recurrent_dropout=0.1))

# Final LSTM layer without returning sequences

model.add(LSTM(32, dropout=0.1, recurrent_dropout=0.1))


# Optional: Layer Normalization (can help with training stability)

model.add(LayerNormalization())

# Optionally reduce or remove additional dropout
model.add(Dropout(0.2))

# Optionally add a dense hidden layer to learn more complex combinations
model.add(Dense(64, activation='relu'))
# Final Dense layer for classification
model.add(Dense(y.shape[1], activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()



In [ ]:
trainable_count = np.sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
print("Trainable parameters:", trainable_count)


In [ ]:
# ----------------------------------------------------------------
# Split data into training and validation sets and set up Early Stopping
# ----------------------------------------------------------------
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Train the model with EarlyStopping
history = model.fit(X_train, y_train,
                    epochs=50,
                    batch_size=8,
                    validation_data=(X_val, y_val),
                    callbacks=[early_stop],
                    verbose=1)


In [ ]:
# ----------------------------------------------------------------
# Print and Plot Train vs. Validation Accuracy
# ----------------------------------------------------------------
train_acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
print("Training Accuracy per Epoch:", train_acc)
print("Validation Accuracy per Epoch:", val_acc)

plt.figure(figsize=(8, 4))
plt.plot(train_acc, label='Train Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training vs. Validation Accuracy')
plt.legend()
plt.show()

In [ ]:
# ----------------------------------------------------------------
# Print and Plot Train vs. Validation Loss
# ----------------------------------------------------------------
train_loss = history.history['loss']
val_loss = history.history['val_loss']
print("Training Loss per Epoch:", train_loss)
print("Validation Loss per Epoch:", val_loss)

plt.figure(figsize=(8, 4))
plt.plot(train_loss, label='Train Loss')
plt.plot(val_loss, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs. Validation Loss')
plt.legend()
plt.show()

In [ ]:
# ----------------------------------------------------------------
# (Optional) Plot a Zoomable Confusion Matrix using Plotly
# ----------------------------------------------------------------
# Predict on the validation set
import plotly.graph_objects as go
y_val_pred = model.predict(X_val)
y_pred = np.argmax(y_val_pred, axis=1)
y_true = np.argmax(y_val, axis=1)
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true, y_pred)

fig = go.Figure(data=go.Heatmap(
    z=cm,
    x=le.classes_,  # Predicted labels
    y=le.classes_,  # True labels
    colorscale='Blues',
    hoverongaps=False,
    text=cm,
    texttemplate="%{text}"
))
fig.update_layout(
    title='Zoomable Confusion Matrix',
    xaxis_title='Predicted Label',
    yaxis_title='True Label',
    xaxis=dict(autorange="reversed")
)
fig.show()

In [ ]:
# Example new set of symptoms for prediction
new_symptoms = "head_ache, vomiting, chest_pain"  #input("Enter your symptoms : ")

# Preprocess the new symptoms (using the same function as before)
def process_new_symptoms(symptom_str):
    tokens = symptom_str.lower().split(',')
    tokens = [t.strip().replace(' ', '_') for t in tokens]
    return tokens

test_tokens = process_new_symptoms(new_symptoms)
# Convert tokens to sequences using the same tokenizer used for training
test_seq = tokenizer.texts_to_sequences(test_tokens)
# Flatten the sequence
test_seq = [idx for sublist in test_seq for idx in sublist]
# Pad the sequence (ensure same dtype as training data)
test_seq = pad_sequences([test_seq], maxlen=max_length, padding='post', dtype='int32')

# Predict probabilities using the model
predictions = model.predict(test_seq)



In [ ]:
#top disease

top2_indices = np.argsort(predictions[0])[-1:][::-1]
top2_diseases = le.inverse_transform(top2_indices)
predicted_disease = top2_diseases[0]
predicted_disease


In [ ]:
symptoms = pd.read_csv("symtoms_df.csv")
precautions = pd.read_csv("precautions_df.csv")
workout = pd.read_csv("workout_df.csv")
description = pd.read_csv("description.csv")
medications = pd.read_csv('medications.csv')
diets = pd.read_csv("diets.csv")

In [ ]:
def mapping(sample_dis):
    desc = description[description['Disease'] == sample_dis]['Description']
    desc = " ".join([w for w in desc])

    pre = precautions[precautions['Disease'] == sample_dis][['Precaution_1', 'Precaution_2', 'Precaution_3', 'Precaution_4']]
    pre = [col for col in pre.values]

    med = medications[medications['Disease'] == sample_dis]['Medication']
    med = [med for med in med.values]

    die = diets[diets['Disease'] == sample_dis]['Diet']
    die = [die for die in die.values]

    wrkout = workout[workout['disease'] == sample_dis] ['workout']


    return desc,pre,med,die,wrkout

In [ ]:
desc, pre, med, die, wrkout = mapping(predicted_disease)

print("predicted disease:")
print(predicted_disease)
print("description:")
print(desc)

print("precautions:")
for i in pre:
    print(i)   

print("medications:")
for i in med:
    print( i)
    

print("workout:")
for i in wrkout:
    print(i)
    

print("diets:")
for i in die:
    print(i)
    

In [ ]:
# Save the trained model
model.save('medical_model.h5')

# Save the tokenizer
import pickle
with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

# Save the label encoder
with open('label_encoder.pickle', 'wb') as handle:
    pickle.dump(le, handle, protocol=pickle.HIGHEST_PROTOCOL)

print("Model and resources saved successfully!")

In [ ]:
'''



# Step 1: Save the complete model
# This saves the model architecture, weights, optimizer state, and compilation information
model.save('medical_diagnosis_model.h5')
print("Complete model saved to 'medical_diagnosis_model.h5'")

# Step 2: Save the tokenizer
# The tokenizer is essential for preprocessing new symptoms text
import pickle

# Save the tokenizer
with open('medical_tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)
print("Tokenizer saved to 'medical_tokenizer.pickle'")

# Step 3: Save the label encoder
# The label encoder is needed to convert model outputs back to disease names
with open('medical_label_encoder.pickle', 'wb') as handle:
    pickle.dump(le, handle, protocol=pickle.HIGHEST_PROTOCOL)
print("Label encoder saved to 'medical_label_encoder.pickle'")

# Step 4: Save essential preprocessing information
import json

# Save max_length for padding
preprocessing_info = {
    'max_length': max_length,
    'embedding_dim': embedding_dim,
    'vocab_size': vocab_size
}

with open('preprocessing_info.json', 'w') as f:
    json.dump(preprocessing_info, f)
print("Preprocessing information saved to 'preprocessing_info.json'")

# Step 5: Save the symptom severity dictionary
with open('severity_dict.pickle', 'wb') as handle:
    pickle.dump(severity_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)
print("Symptom severity dictionary saved to 'severity_dict.pickle'")

# Step 6: Create a simple function to demonstrate loading and using the model
def create_prediction_function():
    with open('prediction_function.py', 'w') as f:
        f.write("""
import numpy as np
import tensorflow as tf
import pickle
import json
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Custom layer is required for loading the model
class WeightedEmbedding(tf.keras.layers.Layer):
    def __init__(self, input_dim, output_dim, embedding_scaling, input_length, **kwargs):
        super(WeightedEmbedding, self).__init__(**kwargs)
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.input_length = input_length
        
        if isinstance(embedding_scaling, tf.Tensor):
            self.embedding_scaling = embedding_scaling.numpy().tolist()
        else:
            self.embedding_scaling = embedding_scaling

        self.embedding = tf.keras.layers.Embedding(
            input_dim=self.input_dim,
            output_dim=self.output_dim,
            input_length=self.input_length,
            mask_zero=False
        )
    
    def call(self, inputs):
        inputs = tf.cast(inputs, tf.int32)
        embeddings = self.embedding(inputs)
        scaling_tensor = tf.convert_to_tensor(self.embedding_scaling, dtype=tf.float32)
        weights = tf.gather(scaling_tensor, inputs)
        weights = tf.expand_dims(weights, axis=-1)
        weighted_embeddings = embeddings * weights
        return weighted_embeddings
    
    def compute_output_shape(self, input_shape):
        return input_shape + (self.output_dim,)
    
    def get_config(self):
        config = super(WeightedEmbedding, self).get_config()
        config.update({
            'input_dim': self.input_dim,
            'output_dim': self.output_dim,
            'embedding_scaling': self.embedding_scaling,
            'input_length': self.input_length,
        })
        return config

def load_model_and_resources():
    # Load the model with custom layer
    model = tf.keras.models.load_model('medical_diagnosis_model.h5', 
                                       custom_objects={'WeightedEmbedding': WeightedEmbedding})
    
    # Load the tokenizer
    with open('medical_tokenizer.pickle', 'rb') as handle:
        tokenizer = pickle.load(handle)
    
    # Load the label encoder
    with open('medical_label_encoder.pickle', 'rb') as handle:
        label_encoder = pickle.load(handle)
    
    # Load preprocessing info
    with open('preprocessing_info.json', 'r') as f:
        preprocessing_info = json.load(f)
    
    return model, tokenizer, label_encoder, preprocessing_info

def process_symptoms(symptom_str):
    '''Process symptoms string into tokens'''
    tokens = symptom_str.lower().split(',')
    tokens = [t.strip().replace(' ', '_') for t in tokens]
    return tokens

def predict_disease(symptoms_text):
    '''Predict disease from symptoms text'''
    # Load model and resources
    model, tokenizer, label_encoder, preprocessing_info = load_model_and_resources()
    
    # Process the symptoms
    tokens = process_symptoms(symptoms_text)
    
    # Convert tokens to sequences
    sequences = tokenizer.texts_to_sequences(tokens)
    
    # Flatten the sequences
    flattened_seq = [idx for sublist in sequences for idx in sublist]
    
    # Pad the sequence
    padded_seq = pad_sequences([flattened_seq], 
                               maxlen=preprocessing_info['max_length'], 
                               padding='post', 
                               dtype='int32')
    
    # Make prediction
    predictions = model.predict(padded_seq)
    
    # Get top disease
    top_index = np.argmax(predictions[0])
    predicted_disease = label_encoder.inverse_transform([top_index])[0]
    
    # Get confidence score
    confidence = float(predictions[0][top_index])
    
    return {
        'disease': predicted_disease,
        'confidence': confidence,
        'top_predictions': get_top_n_predictions(predictions[0], label_encoder, n=3)
    }

def get_top_n_predictions(prediction_array, label_encoder, n=3):
    '''Get top N predictions with their confidence scores'''
    top_indices = np.argsort(prediction_array)[-n:][::-1]
    result = []
    
    for idx in top_indices:
        disease = label_encoder.inverse_transform([idx])[0]
        confidence = float(prediction_array[idx])
        result.append({
            'disease': disease,
            'confidence': confidence
        })
    
    return result

# Example usage
if __name__ == "__main__":
    # Example symptoms
    sample_symptoms = "Nausea, vomiting, chest_pain"
    
    # Predict disease
    result = predict_disease(sample_symptoms)
    
    # Print results
    print(f"Predicted Disease: {result['disease']}")
    print(f"Confidence: {result['confidence']:.2f}")
    print("\\nTop 3 Predictions:")
    for pred in result['top_predictions']:
        print(f"- {pred['disease']}: {pred['confidence']:.2f}")
""")
    print("Prediction function saved to 'prediction_function.py'")

# Create the prediction function file
create_prediction_function()

# Step 7: Create a simple Flask API for deployment
def create_flask_app():
    with open('app.py', 'w') as f:
        f.write("""
from flask import Flask, request, jsonify
from prediction_function import predict_disease

app = Flask(__name__)

@app.route('/predict', methods=['POST'])
def predict():
    data = request.get_json()
    
    if 'symptoms' not in data:
        return jsonify({'error': 'No symptoms provided'}), 400
    
    symptoms = data['symptoms']
    
    try:
        result = predict_disease(symptoms)
        return jsonify(result)
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/health', methods=['GET'])
def health_check():
    return jsonify({'status': 'healthy'})

if __name__ == '__main__':
    app.run(debug=True, host='0.0.0.0', port=5000)
""")
    print("Flask API created in 'app.py'")

# Create the Flask app file
create_flask_app()

# Step 8: Create requirements.txt for deployment
with open('requirements.txt', 'w') as f:
    f.write("""
tensorflow>=2.0.0
numpy>=1.19.2
flask>=2.0.0
gunicorn>=20.0.4
scikit-learn>=0.24.0
pandas>=1.1.3
""")
print("Requirements file created in 'requirements.txt'")

# Step 9: Create a simple README file with deployment instructions
with open('README.md', 'w') as f:
    f.write("""# Medical Diagnosis Model Deployment

This repository contains a medical diagnosis model that predicts diseases based on symptoms.

## Files
- `medical_diagnosis_model.h5`: The trained TensorFlow model
- `medical_tokenizer.pickle`: The tokenizer for processing symptoms text
- `medical_label_encoder.pickle`: The label encoder for converting predictions to disease names
- `preprocessing_info.json`: Contains preprocessing parameters
- `severity_dict.pickle`: Dictionary of symptom severity weights
- `prediction_function.py`: Utility function for making predictions
- `app.py`: Flask API for serving predictions
- `requirements.txt`: Required Python packages

## Deployment Instructions

### Local Deployment
1. Install requirements:
   ```
   pip install -r requirements.txt
   ```

2. Run the Flask application:
   ```
   python app.py
   ```

3. Test the API:
   ```
   curl -X POST http://localhost:5000/predict \\
   -H "Content-Type: application/json" \\
   -d '{"symptoms": "Nausea, vomiting, chest_pain"}'
   ```

### Docker Deployment
1. Build Docker image:
   ```
   docker build -t medical-diagnosis-api .
   ```

2. Run Docker container:
   ```
   docker run -p 5000:5000 medical-diagnosis-api
   ```

### Cloud Deployment (AWS)
1. Package all files
2. Upload to AWS Elastic Beanstalk or set up on EC2
3. Configure security groups to allow traffic on port 5000
4. Set up a load balancer if needed for high availability

## Usage Example

```python
import requests

response = requests.post('http://localhost:5000/predict', 
                        json={'symptoms': 'Nausea, vomiting, chest_pain'})
                        
print(response.json())
```
""")
print("README.md created with deployment instructions")

# Step 10: Create a simple Dockerfile
with open('Dockerfile', 'w') as f:
    f.write("""FROM python:3.9-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 5000

CMD ["gunicorn", "--bind", "0.0.0.0:5000", "app:app"]
""")
print("Dockerfile created for containerization")

print("\nAll files for model deployment have been created successfully!")




'''